# 01 — Basic EDA

最初の目的は「きれいなグラフを作ること」ではなく、データ契約を確認し、
学習パイプラインを壊す要因を早く見つけることです。

ここでは次を確認します。

1. train/test の行数と列の対応
2. ID の一意性と目的変数の分布
3. dtype、欠損率、ユニーク数
4. 数値・カテゴリ特徴量の役割
5. 各特徴量の分布と `Churn=Yes` の比率

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from features import get_base_features

pd.set_option("display.max_columns", 100)
INPUT_DIR = ROOT / "input"
TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN

train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")
print("ROOT :", ROOT)
print("train:", train.shape, "test:", test.shape)
display(train.head())

## データ契約

以下のassertが失敗した場合、モデリング前に入力ファイルを確認します。

In [ ]:
assert TARGET in train.columns and TARGET not in test.columns
assert train[ID_COLUMN].is_unique and test[ID_COLUMN].is_unique
assert set(train.columns) - {TARGET} == set(test.columns)
assert train[TARGET].notna().all()
print("data contract: OK")

## 目的変数

ROC AUCを使うため、クラス比率と欠損を確認します。

In [ ]:
target_summary = (
    train[TARGET]
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .to_frame("count")
)
target_summary["rate"] = target_summary["count"] / len(train)
display(target_summary)

ax = target_summary["count"].plot.pie(
    title="Target distribution", autopct="%1.1f%%", startangle=90
)
ax.set_ylabel("")
ax.axis("equal")
plt.show()

## スキーマと欠損

train/testで欠損率が大きく違う列は、分布シフトの候補です。

In [ ]:
feature_columns = [column for column in train.columns if column != TARGET]
schema = pd.DataFrame(
    {
        "dtype": train[feature_columns].dtypes.astype(str),
        "train_missing_n": train[feature_columns].isna().sum(),
        "train_missing_rate": train[feature_columns].isna().mean(),
        "test_missing_n": test[feature_columns].isna().sum(),
        "test_missing_rate": test[feature_columns].isna().mean(),
        "train_nunique": train[feature_columns].nunique(dropna=False),
        "test_nunique": test[feature_columns].nunique(dropna=False),
    }
)
schema["missing_rate_gap"] = (
    schema["train_missing_rate"] - schema["test_missing_rate"]
).abs()
display(schema.sort_values(["train_missing_rate", "missing_rate_gap"], ascending=False))

## モデル上の列役割

dtypeだけで機械的に決めると、0/1で表現されたカテゴリ列を連続値として扱うことがあります。
ここでは数値dtypeかつユニーク数が5以上の列だけを数値特徴量にします。

In [ ]:
BASE_FEATURES, BASE_NUM, BASE_CAT = get_base_features(
    train, TARGET, ID_COLUMN, min_nunique=Baseline.MIN_NUMERIC_UNIQUE
)
print("numeric    :", BASE_NUM)
print("categorical:", BASE_CAT)
display(train[BASE_NUM].describe().T)
display(train[BASE_CAT].describe(include="all").T)

## 特徴量と目的変数の関係

棒が件数、折れ線が各bucketの解約率です。少数bucketは率が不安定なので件数と同時に読みます。

In [ ]:
def plot_features_vs_target(
    frame,
    columns=BASE_FEATURES,
    target=TARGET,
    positive="Yes",
    columns_per_row=3,
):
    rows = int(np.ceil(len(columns) / columns_per_row))
    figure, axes = plt.subplots(
        rows, columns_per_row, figsize=(6 * columns_per_row, 4 * rows)
    )
    axes = np.atleast_1d(axes).ravel()

    for axis, column in zip(axes, columns):
        values = frame[column]
        if column in BASE_NUM:
            bucket = pd.qcut(values, q=10, duplicates="drop").astype("object")
        else:
            bucket = values.astype("object")
        bucket = bucket.fillna("Missing")

        plot_data = (
            pd.DataFrame(
                {"bucket": bucket, "positive": frame[target].eq(positive)}
            )
            .groupby("bucket", observed=False)
            .agg(count=("positive", "size"), positive_rate=("positive", "mean"))
            .reset_index()
        )
        positions = np.arange(len(plot_data))
        axis.bar(positions, plot_data["count"], alpha=0.7)
        axis.set_title(column)
        axis.set_ylabel("Count")
        axis.set_xticks(positions)
        axis.set_xticklabels(plot_data["bucket"].astype(str), rotation=45, ha="right")

        rate_axis = axis.twinx()
        rate_axis.plot(positions, plot_data["positive_rate"], marker="o", color="tomato")
        rate_axis.set_ylim(0, 1)
        rate_axis.set_ylabel(f"{positive} rate")
        rate_axis.yaxis.set_major_formatter(PercentFormatter(1))

    for axis in axes[len(columns):]:
        axis.set_visible(False)
    figure.suptitle(f"Feature distributions and {target}={positive} rate", y=1.01)
    plt.tight_layout()
    plt.show()


plot_features_vs_target(train)

## 次へ進む判断

- IDは識別子なので特徴量から除外します。
- 欠損補完とカテゴリ変換は、必ずCV foldの内側でfitします。
- ここで見えた関係は仮説であり、採用判断はOOF AUCで行います。